In [1]:
# ============================================================
# 15_universities_v8.ipynb
# Domain: universities
# Goal: 95 high-quality RU/EN multihop benchmark queries for universities.
#
# v8 fixes compared to v7:
#   * removes trivial country-only L1 records from the default target;
#   * removes all user-facing "excluding / не включая" constraints;
#   * forbids redundant country+capital criteria in one query;
#   * upgrades L3-L5 with real multihop criteria: Nobel-laureate alumni,
#     founding-year comparisons, university-association membership,
#     capital-derived country and same administrative entity patterns;
#   * keeps constraints clean and human-readable; QIDs remain in metadata.


In [2]:
# ============================================================
from __future__ import annotations

from pathlib import Path
from dataclasses import asdict, fields
from typing import Any, Callable, Dict, Iterable, List, Optional, Sequence, Tuple
from collections import Counter
import json
import random
import re
import time


In [3]:
# ============================================================
# 0. Load shared project helpers


In [4]:
# ============================================================
# The project uses common_helpers.py as the executable version of
# 00_common_helpers.ipynb. Loading it preserves the exact BenchmarkExample
# dataclass and WikidataClient behavior used by other domains.
if "BenchmarkExample" not in globals():
    exec(Path("common_helpers.py").read_text(encoding="utf-8"), globals())

try:
    import pandas as pd
except Exception as e:
    raise RuntimeError("pandas is required by common_helpers.py and this notebook") from e

WD_CLIENT = globals().get("wd")
if WD_CLIENT is None or not hasattr(WD_CLIENT, "sparql_select"):
    raise RuntimeError("common_helpers.py did not initialize Wikidata client `wd` correctly")

print("✅ common helpers loaded; WD_CLIENT captured")


✅ Patched: WikidataClient.sparql_select (robust) + load_or_build_pool (safe)
✅ Patched: select_items_with_* используют ru/en fallback + repair_pool_labels чинит QID вместо label
✅ common helpers loaded; WD_CLIENT captured


/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# ============================================================
# 1. Domain configuration


In [6]:
# ============================================================
DOMAIN = "universities"
VERSION = "v8"
Q_UNIVERSITY = "Q3918"  # university
Q_NOBEL_PRIZE = "Q7191"  # Nobel Prize

DOMAIN_OUTPUT_DIR = Path("out_wikidata_benchmark/domain_outputs")
DOMAIN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATH = DOMAIN_OUTPUT_DIR / "universities.jsonl"

# User-requested distribution: 95 records total.
# L3+ are intentionally overrepresented because this domain is for multihop benchmarking.
TARGET_PER_LEVEL: Dict[str, int] = {
    # L1 country-only records were too easy for this multihop benchmark,
    # so v8 does not generate them by default.
    "L1": 0,
    "L2": 15,
    "L3": 25,
    "L4": 25,
    "L5": 30,
}
LEVELS: List[str] = ["L1", "L2", "L3", "L4", "L5"]

REQUESTED_BY_LEVEL: Dict[str, int] = {
    "L1": 5,
    "L2": 5,
    "L3": 4,
    "L4": 3,
    "L5": 3,
}

MAX_GOLD_BY_LEVEL: Dict[str, int] = {
    "L1": 220,
    "L2": 180,
    "L3": 140,
    "L4": 110,
    "L5": 90,
}

# Direct P31=university gives much cleaner gold than P31/P279*=university.
# It avoids many faculties, departments, schools, campuses and other higher-education
# related entities that are not natural answers to "universities".
DIRECT_INSTANCE_ONLY = True

SEED = 20260614
MAX_ATTEMPTS_PER_LEVEL = 30000
OVERWRITE_OUTPUT = True
RUN_GENERATION = True
RUN_FINAL_VALIDATION = True
DEBUG_GENERATOR_ERRORS = False
DEBUG_REJECTIONS = False

print(f"✅ config ready [{DOMAIN} {VERSION}] target={TARGET_PER_LEVEL}, direct_instance_only={DIRECT_INSTANCE_ONLY}")


✅ config ready [universities v8] target={'L1': 0, 'L2': 15, 'L3': 25, 'L4': 25, 'L5': 30}, direct_instance_only=True


In [7]:
# ============================================================
# 2. Schema and cleaning helpers


In [8]:
# ============================================================
EXPECTED_KEYS: List[str] = [f.name for f in fields(BenchmarkExample)]
CONTROL_RE = re.compile(r"[\u200e\u200f\u202a-\u202e\ufeff]")
CYRILLIC_RE = re.compile(r"[А-Яа-яЁё]")
QID_ONLY_RE = re.compile(r"^Q\d+$")
PID_ONLY_RE = re.compile(r"^P\d+$")
CONSTRAINT_META_KEY_RE = re.compile(r"(^|_)(qid|pid|wikidata|wd|sparql)($|_)", re.I)
BAD_QUERY_PHRASES_RE = re.compile(r"Wikidata|official website|coordinate locations?|координат|официальн\w+ сайт|викидан|не включая|исключая|excluding|exclude_anchor", re.I)

LAST_REJECT_REASON = ""


def clean_string(value: Any) -> str:
    value = CONTROL_RE.sub("", str(value or ""))
    value = re.sub(r"\s+", " ", value).strip()
    return value


def clean_text(value: Any) -> Any:
    if isinstance(value, str):
        return clean_string(value)
    if isinstance(value, list):
        return [clean_text(v) for v in value]
    if isinstance(value, tuple):
        return [clean_text(v) for v in value]
    if isinstance(value, dict):
        return {str(k): clean_text(v) for k, v in value.items() if v is not None}
    return value


def has_cyrillic(value: Any) -> bool:
    if isinstance(value, str):
        return bool(CYRILLIC_RE.search(value))
    if isinstance(value, list):
        return any(has_cyrillic(v) for v in value)
    if isinstance(value, dict):
        return any(has_cyrillic(k) or has_cyrillic(v) for k, v in value.items())
    return False


def has_qid_or_pid(value: Any) -> bool:
    if isinstance(value, str):
        s = clean_string(value)
        return bool(QID_ONLY_RE.fullmatch(s) or PID_ONLY_RE.fullmatch(s) or re.search(r"\b[QP]\d+\b", s))
    if isinstance(value, list):
        return any(has_qid_or_pid(v) for v in value)
    if isinstance(value, dict):
        return any(has_qid_or_pid(k) or has_qid_or_pid(v) for k, v in value.items())
    return False


def qid_from_any(value: Any) -> Optional[str]:
    try:
        return uri_to_qid(str(value or ""))
    except Exception:
        m = re.search(r"Q\d+", str(value or ""))
        return m.group(0) if m else None


def label_ok(label: Any) -> bool:
    label = clean_string(label)
    if not label:
        return False
    if QID_ONLY_RE.fullmatch(label) or PID_ONLY_RE.fullmatch(label):
        return False
    return True


def first_good_label(*labels: Any) -> str:
    for label in labels:
        if label_ok(label):
            return clean_string(label)
    return ""


def ru_name(label_ru: Any, label_en: Any = "") -> str:
    return first_good_label(label_ru, label_en)


def en_name(label_en: Any, label_ru: Any = "") -> str:
    return first_good_label(label_en, label_ru)


def ordered_as_benchmark_example(ex: BenchmarkExample) -> Dict[str, Any]:
    return asdict(ex)


def constraints_are_clean(constraints: Dict[str, Any]) -> bool:
    if not isinstance(constraints, dict):
        return False

    # Avoid the redundant pattern criticized in v7: deriving the country from
    # an anchor university and also restating the capital of that same country.
    # Each query should have at most one user-facing country/city criterion.
    if "country_capital" in constraints and ("country" in constraints or "country_from_university" in constraints):
        return False

    for key, value in constraints.items():
        if not isinstance(key, str):
            return False
        if "exclude" in key.lower():
            return False
        if CONSTRAINT_META_KEY_RE.search(key):
            return False
        if has_cyrillic(key) or has_cyrillic(value):
            return False
        if isinstance(value, dict):
            return False
        if has_qid_or_pid(key) or has_qid_or_pid(value):
            return False
        if isinstance(value, str) and BAD_QUERY_PHRASES_RE.search(value):
            return False
    return True


def reject(reason: str) -> None:
    global LAST_REJECT_REASON
    LAST_REJECT_REASON = reason
    if DEBUG_REJECTIONS:
        print("[reject]", reason)


def read_jsonl(path: Path) -> List[Dict[str, Any]]:
    if not path.exists():
        return []
    rows: List[Dict[str, Any]] = []
    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as e:
                raise RuntimeError(f"Bad JSONL line {line_no} in {path}: {e}") from e
    return rows


def append_example_jsonl(path: Path, ex: BenchmarkExample) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(ordered_as_benchmark_example(ex), ensure_ascii=False) + "\n")


def ru_university_word(k: int) -> str:
    k = abs(int(k))
    if 10 <= k % 100 <= 20:
        return "университетов"
    if k % 10 == 1:
        return "университет"
    if 2 <= k % 10 <= 4:
        return "университета"
    return "университетов"


def question_text_ok(text: str) -> bool:
    return bool(clean_string(text)) and BAD_QUERY_PHRASES_RE.search(text) is None

print("✅ schema helpers ready; clean constraints forbid qid/pid/wikidata/property-existence fields")


✅ schema helpers ready; clean constraints forbid qid/pid/wikidata/property-existence fields


In [9]:
# ============================================================
# 3. SPARQL builders


In [10]:
# ============================================================

def wd_entity(qid: str) -> str:
    qid = clean_string(qid)
    if not re.fullmatch(r"Q\d+", qid):
        raise ValueError(f"Not a QID: {qid}")
    return f"wd:{qid}"


def class_membership_line(item_var: str = "item", direct_only: bool = DIRECT_INSTANCE_ONLY) -> str:
    if direct_only:
        return f"?{item_var} wdt:P31 wd:{Q_UNIVERSITY} ."
    return f"?{item_var} wdt:P31/wdt:P279* wd:{Q_UNIVERSITY} ."


def label_block(item_var: str = "item") -> str:
    return f'''OPTIONAL {{ ?{item_var} rdfs:label ?{item_var}LabelRu FILTER(LANG(?{item_var}LabelRu) = "ru") . }}
      OPTIONAL {{ ?{item_var} rdfs:label ?{item_var}LabelEn FILTER(LANG(?{item_var}LabelEn) = "en") . }}
      FILTER(BOUND(?{item_var}LabelRu) || BOUND(?{item_var}LabelEn)) .'''


def dedupe_where_lines(where_lines: Sequence[str]) -> List[str]:
    out: List[str] = []
    seen = set()
    for line in where_lines:
        line = str(line).rstrip()
        if not line or line in seen:
            continue
        seen.add(line)
        out.append(line)
    return out


def build_select_sparql(where_lines: Sequence[str], *, item_var: str = "item", limit: int = 100, direct_only: bool = DIRECT_INSTANCE_ONLY) -> str:
    where = "\n      ".join(dedupe_where_lines(where_lines))
    return f'''
    SELECT DISTINCT ?{item_var} ?{item_var}LabelRu ?{item_var}LabelEn WHERE {{
      {class_membership_line(item_var=item_var, direct_only=direct_only)}
      {where}
      {label_block(item_var=item_var)}
    }}
    LIMIT {int(limit)}
    '''.strip()


def build_ask_sparql(where_lines: Sequence[str], *, item_var: str = "item", direct_only: bool = DIRECT_INSTANCE_ONLY) -> str:
    where = "\n      ".join(dedupe_where_lines(where_lines))
    return f'''
    ASK WHERE {{
      BIND(wd:{{ITEM}} AS ?{item_var})
      {class_membership_line(item_var=item_var, direct_only=direct_only)}
      {where}
    }}
    '''.strip()


def year_filter_lines(*, min_year: Optional[int] = None, max_year: Optional[int] = None, var: str = "inception", year_var: str = "year") -> List[str]:
    lines = [
        f"?item wdt:P571 ?{var} .",
        f"BIND(YEAR(?{var}) AS ?{year_var}) .",
    ]
    if min_year is not None and max_year is not None:
        lines.append(f"FILTER(?{year_var} >= {int(min_year)} && ?{year_var} <= {int(max_year)}) .")
    elif min_year is not None:
        lines.append(f"FILTER(?{year_var} >= {int(min_year)}) .")
    elif max_year is not None:
        lines.append(f"FILTER(?{year_var} <= {int(max_year)}) .")
    return lines


def inception_window_lines(y1: int, y2: int) -> List[str]:
    return year_filter_lines(min_year=int(y1), max_year=int(y2))


def before_year_lines(year: int) -> List[str]:
    return year_filter_lines(max_year=int(year) - 1)


def after_year_lines(year: int) -> List[str]:
    return year_filter_lines(min_year=int(year) + 1)


def exclude_qid_line(qid: str) -> str:
    return f"FILTER(?item != wd:{qid}) ."


def same_country_lines(anchor_qid: str, *, exclude_anchor: bool = True) -> List[str]:
    lines = [
        f"wd:{anchor_qid} wdt:P17 ?country .",
        "?item wdt:P17 ?country .",
    ]
    if exclude_anchor:
        lines.append(exclude_qid_line(anchor_qid))
    return lines


def same_admin_area_lines(anchor_qid: str, *, exclude_anchor: bool = True) -> List[str]:
    lines = [
        f"wd:{anchor_qid} wdt:P131 ?admin_area .",
        "?item wdt:P131 ?admin_area .",
    ]
    if exclude_anchor:
        lines.append(exclude_qid_line(anchor_qid))
    return lines


def same_membership_lines(anchor_qid: str, *, exclude_anchor: bool = True) -> List[str]:
    lines = [
        f"wd:{anchor_qid} wdt:P463 ?membership .",
        "?item wdt:P463 ?membership .",
    ]
    if exclude_anchor:
        lines.append(exclude_qid_line(anchor_qid))
    return lines

print("✅ SPARQL builders ready")


✅ SPARQL builders ready


In [11]:
# ============================================================
# 4. Gold collection and example construction


In [12]:
# ============================================================

def max_gold(level: str) -> int:
    return int(MAX_GOLD_BY_LEVEL[level])


def gold_query_limit(level: str) -> int:
    return max_gold(level) + 1


def rows_to_gold(rows: List[Dict[str, str]], *, item_var: str = "item") -> Tuple[List[str], List[str], List[str]]:
    qids: List[str] = []
    labels_ru: List[str] = []
    labels_en: List[str] = []
    seen = set()
    for row in rows:
        qid = qid_from_any(row.get(item_var))
        if not qid or qid in seen:
            continue
        label_ru = first_good_label(row.get(f"{item_var}LabelRu"), row.get(f"{item_var}LabelEn"))
        label_en = first_good_label(row.get(f"{item_var}LabelEn"), row.get(f"{item_var}LabelRu"))
        if not label_ok(label_ru) or not label_ok(label_en):
            continue
        seen.add(qid)
        qids.append(qid)
        labels_ru.append(label_ru)
        labels_en.append(label_en)
    return qids, labels_ru, labels_en


def collect_gold(where_lines: Sequence[str], *, level: str, item_var: str = "item") -> Tuple[str, List[str], List[str], List[str], bool]:
    sparql = build_select_sparql(where_lines, item_var=item_var, limit=gold_query_limit(level))
    data = WD_CLIENT.sparql_select(sparql)
    rows = rows_from_select(data)
    qids, labels_ru, labels_en = rows_to_gold(rows, item_var=item_var)
    truncated = len(qids) > max_gold(level)
    if truncated:
        qids = qids[:max_gold(level)]
        labels_ru = labels_ru[:max_gold(level)]
        labels_en = labels_en[:max_gold(level)]
    return sparql, qids, labels_ru, labels_en, truncated


def base_meta(template_id: str, constraint_entity_qids: Optional[Dict[str, Any]] = None, derived_labels: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
    return clean_text({
        "source": "wikidata_wdqs",
        "generator_version": f"universities_{VERSION}",
        "template_id": template_id,
        "direct_instance_only": DIRECT_INSTANCE_ONLY,
        "constraint_entity_qids": constraint_entity_qids or {},
        "derived_labels": derived_labels or {},
        "note": "Entity identifiers used to execute SPARQL are stored here so constraints stay clean and human-readable.",
    })


def make_example(
    *,
    level: str,
    idx: int,
    query_text_ru: str,
    query_text_en: str,
    constraints: Dict[str, Any],
    where_lines: Sequence[str],
    template_id: str,
    template_family: str,
    is_advanced: bool,
    constraint_entity_qids: Optional[Dict[str, Any]] = None,
    derived_labels: Optional[Dict[str, Any]] = None,
) -> Optional[BenchmarkExample]:
    requested = int(REQUESTED_BY_LEVEL[level])
    constraints = clean_text(constraints)

    if not question_text_ok(query_text_ru) or not question_text_ok(query_text_en):
        reject(f"bad user-facing query text for {template_id}")
        return None
    if not constraints_are_clean(constraints):
        reject(f"bad constraints for {template_id}: {constraints}")
        return None

    try:
        sparql, qids, labels_ru, labels_en, truncated = collect_gold(where_lines, level=level)
    except Exception as e:
        reject(f"gold_query_failed:{template_id}:{e}")
        if DEBUG_GENERATOR_ERRORS:
            print(f"[WARN] gold query failed for {template_id}: {e}")
        return None

    if truncated:
        reject(f"truncated_gold:{template_id}:>{max_gold(level)}")
        return None
    if len(qids) < requested:
        reject(f"too_few_gold:{template_id}:{len(qids)}<{requested}")
        return None

    ask = build_ask_sparql(where_lines)
    if "wd:{ITEM}" not in ask:
        reject(f"bad_ask_validator:{template_id}")
        return None

    ex = BenchmarkExample(
        id=f"{DOMAIN}_{level.lower()}_{idx:04d}",
        domain=DOMAIN,
        complexity=level,
        query_text_ru=clean_text(query_text_ru),
        constraints=constraints,
        requested_count=requested,
        gold_answer_qids=qids,
        gold_answer_labels_ru=labels_ru,
        sparql_query=sparql,
        created_at=utc_now_z(),
        query_text_en=clean_text(query_text_en),
        gold_answer_labels_en=labels_en,
        is_advanced=bool(is_advanced),
        template_id=template_id,
        template_family=template_family,
        gold_truncated=False,
        ask_validator_sparql=ask,
        gold_collection_meta=base_meta(template_id, constraint_entity_qids, derived_labels),
    )

    if list(ordered_as_benchmark_example(ex).keys()) != EXPECTED_KEYS:
        reject(f"schema_mismatch:{template_id}")
        return None
    return ex

print("✅ gold collection ready; zero/underfilled/truncated gold records rejected")


✅ gold collection ready; zero/underfilled/truncated gold records rejected


In [13]:
# ============================================================
# 5. Static pools


In [14]:
# ============================================================
COUNTRIES: List[Dict[str, str]] = [
    {"qid": "Q30", "en": "United States", "ru": "США"},
    {"qid": "Q145", "en": "United Kingdom", "ru": "Великобритания"},
    {"qid": "Q183", "en": "Germany", "ru": "Германия"},
    {"qid": "Q142", "en": "France", "ru": "Франция"},
    {"qid": "Q159", "en": "Russia", "ru": "Россия"},
    {"qid": "Q17", "en": "Japan", "ru": "Япония"},
    {"qid": "Q148", "en": "China", "ru": "Китай"},
    {"qid": "Q16", "en": "Canada", "ru": "Канада"},
    {"qid": "Q38", "en": "Italy", "ru": "Италия"},
    {"qid": "Q29", "en": "Spain", "ru": "Испания"},
    {"qid": "Q55", "en": "Netherlands", "ru": "Нидерланды"},
    {"qid": "Q408", "en": "Australia", "ru": "Австралия"},
    {"qid": "Q39", "en": "Switzerland", "ru": "Швейцария"},
    {"qid": "Q40", "en": "Austria", "ru": "Австрия"},
    {"qid": "Q36", "en": "Poland", "ru": "Польша"},
    {"qid": "Q34", "en": "Sweden", "ru": "Швеция"},
    {"qid": "Q33", "en": "Finland", "ru": "Финляндия"},
    {"qid": "Q35", "en": "Denmark", "ru": "Дания"},
    {"qid": "Q27", "en": "Ireland", "ru": "Ирландия"},
    {"qid": "Q213", "en": "Czech Republic", "ru": "Чехия"},
    {"qid": "Q28", "en": "Hungary", "ru": "Венгрия"},
    {"qid": "Q41", "en": "Greece", "ru": "Греция"},
    {"qid": "Q43", "en": "Turkey", "ru": "Турция"},
    {"qid": "Q155", "en": "Brazil", "ru": "Бразилия"},
    {"qid": "Q96", "en": "Mexico", "ru": "Мексика"},
    {"qid": "Q414", "en": "Argentina", "ru": "Аргентина"},
    {"qid": "Q298", "en": "Chile", "ru": "Чили"},
    {"qid": "Q739", "en": "Colombia", "ru": "Колумбия"},
    {"qid": "Q668", "en": "India", "ru": "Индия"},
    {"qid": "Q884", "en": "South Korea", "ru": "Южная Корея"},
    {"qid": "Q865", "en": "Taiwan", "ru": "Тайвань"},
    {"qid": "Q212", "en": "Ukraine", "ru": "Украина"},
    {"qid": "Q232", "en": "Kazakhstan", "ru": "Казахстан"},
    {"qid": "Q219", "en": "Bulgaria", "ru": "Болгария"},
    {"qid": "Q218", "en": "Romania", "ru": "Румыния"},
    {"qid": "Q224", "en": "Croatia", "ru": "Хорватия"},
    {"qid": "Q215", "en": "Slovenia", "ru": "Словения"},
    {"qid": "Q214", "en": "Slovakia", "ru": "Словакия"},
    {"qid": "Q211", "en": "Latvia", "ru": "Латвия"},
    {"qid": "Q191", "en": "Estonia", "ru": "Эстония"},
    {"qid": "Q37", "en": "Lithuania", "ru": "Литва"},
    {"qid": "Q45", "en": "Portugal", "ru": "Португалия"},
    {"qid": "Q31", "en": "Belgium", "ru": "Бельгия"},
    {"qid": "Q32", "en": "Luxembourg", "ru": "Люксембург"},
    {"qid": "Q801", "en": "Israel", "ru": "Израиль"},
    {"qid": "Q258", "en": "South Africa", "ru": "Южная Африка"},
    {"qid": "Q117", "en": "Ghana", "ru": "Гана"},
    {"qid": "Q1033", "en": "Nigeria", "ru": "Нигерия"},
]


# Curated country-capital and anchor pools used by L3-L5.
# v6 used a large dynamic anchor query before L3; on real runs it could stall or
# fail before any L3+ records were written. v7 keeps those dynamic helpers as
# optional, but the active L3-L5 templates use this stable curated pool.
# QIDs stay in gold_collection_meta; constraints only expose human-readable labels.
CAPITAL_COUNTRIES: List[Dict[str, Any]] = [
    {"country_qid": "Q145", "country_en": "United Kingdom", "country_ru": "Великобритания", "capital_qid": "Q84", "capital_en": "London", "capital_ru": "Лондон"},
    {"country_qid": "Q55", "country_en": "Netherlands", "country_ru": "Нидерланды", "capital_qid": "Q727", "capital_en": "Amsterdam", "capital_ru": "Амстердам"},
    {"country_qid": "Q40", "country_en": "Austria", "country_ru": "Австрия", "capital_qid": "Q1741", "capital_en": "Vienna", "capital_ru": "Вена"},
    {"country_qid": "Q35", "country_en": "Denmark", "country_ru": "Дания", "capital_qid": "Q1748", "capital_en": "Copenhagen", "capital_ru": "Копенгаген"},
    {"country_qid": "Q33", "country_en": "Finland", "country_ru": "Финляндия", "capital_qid": "Q1757", "capital_en": "Helsinki", "capital_ru": "Хельсинки"},
    {"country_qid": "Q41", "country_en": "Greece", "country_ru": "Греция", "capital_qid": "Q1524", "capital_en": "Athens", "capital_ru": "Афины"},
    {"country_qid": "Q39", "country_en": "Switzerland", "country_ru": "Швейцария", "capital_qid": "Q70", "capital_en": "Bern", "capital_ru": "Берн"},
    {"country_qid": "Q29", "country_en": "Spain", "country_ru": "Испания", "capital_qid": "Q2807", "capital_en": "Madrid", "capital_ru": "Мадрид"},
    {"country_qid": "Q38", "country_en": "Italy", "country_ru": "Италия", "capital_qid": "Q220", "capital_en": "Rome", "capital_ru": "Рим"},
    {"country_qid": "Q16", "country_en": "Canada", "country_ru": "Канада", "capital_qid": "Q1930", "capital_en": "Ottawa", "capital_ru": "Оттава"},
    {"country_qid": "Q96", "country_en": "Mexico", "country_ru": "Мексика", "capital_qid": "Q1489", "capital_en": "Mexico City", "capital_ru": "Мехико"},
    {"country_qid": "Q258", "country_en": "South Africa", "country_ru": "Южная Африка", "capital_qid": "Q3926", "capital_en": "Pretoria", "capital_ru": "Претория"},
    {"country_qid": "Q215", "country_en": "Slovenia", "country_ru": "Словения", "capital_qid": "Q437", "capital_en": "Ljubljana", "capital_ru": "Любляна"},
    {"country_qid": "Q224", "country_en": "Croatia", "country_ru": "Хорватия", "capital_qid": "Q1435", "capital_en": "Zagreb", "capital_ru": "Загреб"},
    {"country_qid": "Q211", "country_en": "Latvia", "country_ru": "Латвия", "capital_qid": "Q1773", "capital_en": "Riga", "capital_ru": "Рига"},
    {"country_qid": "Q37", "country_en": "Lithuania", "country_ru": "Литва", "capital_qid": "Q216", "capital_en": "Vilnius", "capital_ru": "Вильнюс"},
]

STATIC_ANCHORS: List[Dict[str, Any]] = [
    {"qid": "Q35794", "en": "University of Cambridge", "ru": "Кембриджский университет", "country_qid": "Q145", "country_en": "United Kingdom", "country_ru": "Великобритания", "capital_qid": "Q84", "capital_en": "London", "capital_ru": "Лондон", "year": 1209},
    {"qid": "Q34433", "en": "University of Oxford", "ru": "Оксфордский университет", "country_qid": "Q145", "country_en": "United Kingdom", "country_ru": "Великобритания", "capital_qid": "Q84", "capital_en": "London", "capital_ru": "Лондон", "year": 1096},
    {"qid": "Q192775", "en": "University of Glasgow", "ru": "Университет Глазго", "country_qid": "Q145", "country_en": "United Kingdom", "country_ru": "Великобритания", "capital_qid": "Q84", "capital_en": "London", "capital_ru": "Лондон", "year": 1451},
    {"qid": "Q160302", "en": "University of Edinburgh", "ru": "Эдинбургский университет", "country_qid": "Q145", "country_en": "United Kingdom", "country_ru": "Великобритания", "capital_qid": "Q84", "capital_en": "London", "capital_ru": "Лондон", "year": 1582},
    {"qid": "Q156598", "en": "Leiden University", "ru": "Лейденский университет", "country_qid": "Q55", "country_en": "Netherlands", "country_ru": "Нидерланды", "capital_qid": "Q727", "capital_en": "Amsterdam", "capital_ru": "Амстердам", "year": 1575},
    {"qid": "Q214341", "en": "University of Amsterdam", "ru": "Амстердамский университет", "country_qid": "Q55", "country_en": "Netherlands", "country_ru": "Нидерланды", "capital_qid": "Q727", "capital_en": "Amsterdam", "capital_ru": "Амстердам", "year": 1632},
    {"qid": "Q850730", "en": "University of Groningen", "ru": "Университет Гронингена", "country_qid": "Q55", "country_en": "Netherlands", "country_ru": "Нидерланды", "capital_qid": "Q727", "capital_en": "Amsterdam", "capital_ru": "Амстердам", "year": 1614},
    {"qid": "Q165980", "en": "University of Vienna", "ru": "Венский университет", "country_qid": "Q40", "country_en": "Austria", "country_ru": "Австрия", "capital_qid": "Q1741", "capital_en": "Vienna", "capital_ru": "Вена", "year": 1365},
    {"qid": "Q622683", "en": "University of Graz", "ru": "Грацский университет", "country_qid": "Q40", "country_en": "Austria", "country_ru": "Австрия", "capital_qid": "Q1741", "capital_en": "Vienna", "capital_ru": "Вена", "year": 1585},
    {"qid": "Q186285", "en": "University of Copenhagen", "ru": "Копенгагенский университет", "country_qid": "Q35", "country_en": "Denmark", "country_ru": "Дания", "capital_qid": "Q1748", "capital_en": "Copenhagen", "capital_ru": "Копенгаген", "year": 1479},
    {"qid": "Q924265", "en": "Aarhus University", "ru": "Орхусский университет", "country_qid": "Q35", "country_en": "Denmark", "country_ru": "Дания", "capital_qid": "Q1748", "capital_en": "Copenhagen", "capital_ru": "Копенгаген", "year": 1928},
    {"qid": "Q28695", "en": "University of Helsinki", "ru": "Хельсинкский университет", "country_qid": "Q33", "country_en": "Finland", "country_ru": "Финляндия", "capital_qid": "Q1757", "capital_en": "Helsinki", "capital_ru": "Хельсинки", "year": 1640},
    {"qid": "Q501841", "en": "University of Turku", "ru": "Университет Турку", "country_qid": "Q33", "country_en": "Finland", "country_ru": "Финляндия", "capital_qid": "Q1757", "capital_en": "Helsinki", "capital_ru": "Хельсинки", "year": 1920},
    {"qid": "Q319078", "en": "University of Melbourne", "ru": "Мельбурнский университет", "country_qid": "Q408", "country_en": "Australia", "country_ru": "Австралия", "capital_qid": "Q3114", "capital_en": "Canberra", "capital_ru": "Канберра", "year": 1853},
    {"qid": "Q866012", "en": "University of Queensland", "ru": "Квинслендский университет", "country_qid": "Q408", "country_en": "Australia", "country_ru": "Австралия", "capital_qid": "Q3114", "capital_en": "Canberra", "capital_ru": "Канберра", "year": 1909},
    {"qid": "Q734764", "en": "University of New South Wales", "ru": "Университет Нового Южного Уэльса", "country_qid": "Q408", "country_en": "Australia", "country_ru": "Австралия", "capital_qid": "Q3114", "capital_en": "Canberra", "capital_ru": "Канберра", "year": 1949},
    {"qid": "Q547867", "en": "National and Kapodistrian University of Athens", "ru": "Афинский университет", "country_qid": "Q41", "country_en": "Greece", "country_ru": "Греция", "capital_qid": "Q1524", "capital_en": "Athens", "capital_ru": "Афины", "year": 1837},
    {"qid": "Q550263", "en": "University of Piraeus", "ru": "Университет Пирея", "country_qid": "Q41", "country_en": "Greece", "country_ru": "Греция", "capital_qid": "Q1524", "capital_en": "Athens", "capital_ru": "Афины", "year": 1938},
    {"qid": "Q222738", "en": "National Autonomous University of Mexico", "ru": "Национальный автономный университет Мексики", "country_qid": "Q96", "country_en": "Mexico", "country_ru": "Мексика", "capital_qid": "Q1489", "capital_en": "Mexico City", "capital_ru": "Мехико", "year": 1910},
    {"qid": "Q29716", "en": "Universidad Autónoma Nuevo León", "ru": "Автономный университет Нуэво-Леона", "country_qid": "Q96", "country_en": "Mexico", "country_ru": "Мексика", "capital_qid": "Q1489", "capital_en": "Mexico City", "capital_ru": "Мехико", "year": 1933},
]

STATIC_ANCHOR_PAIRS: List[Tuple[Dict[str, Any], Dict[str, Any]]] = [
    (STATIC_ANCHORS[0], STATIC_ANCHORS[2]),
    (STATIC_ANCHORS[0], STATIC_ANCHORS[3]),
    (STATIC_ANCHORS[4], STATIC_ANCHORS[5]),
    (STATIC_ANCHORS[7], STATIC_ANCHORS[8]),
    (STATIC_ANCHORS[9], STATIC_ANCHORS[10]),
    (STATIC_ANCHORS[11], STATIC_ANCHORS[12]),
    (STATIC_ANCHORS[13], STATIC_ANCHORS[15]),
    (STATIC_ANCHORS[16], STATIC_ANCHORS[17]),
    (STATIC_ANCHORS[18], STATIC_ANCHORS[19]),
]

# Extra capital/anchor rows make the Nobel-laureate templates productive and
# diversify L3-L5 beyond the smaller European pool. QIDs are metadata only.
EXTRA_CAPITAL_COUNTRIES: List[Dict[str, Any]] = [
    {"country_qid": "Q30", "country_en": "United States", "country_ru": "США", "capital_qid": "Q61", "capital_en": "Washington, D.C.", "capital_ru": "Вашингтон"},
    {"country_qid": "Q183", "country_en": "Germany", "country_ru": "Германия", "capital_qid": "Q64", "capital_en": "Berlin", "capital_ru": "Берлин"},
    {"country_qid": "Q142", "country_en": "France", "country_ru": "Франция", "capital_qid": "Q90", "capital_en": "Paris", "capital_ru": "Париж"},
    {"country_qid": "Q17", "country_en": "Japan", "country_ru": "Япония", "capital_qid": "Q1490", "capital_en": "Tokyo", "capital_ru": "Токио"},
    {"country_qid": "Q34", "country_en": "Sweden", "country_ru": "Швеция", "capital_qid": "Q1754", "capital_en": "Stockholm", "capital_ru": "Стокгольм"},
    {"country_qid": "Q36", "country_en": "Poland", "country_ru": "Польша", "capital_qid": "Q270", "capital_en": "Warsaw", "capital_ru": "Варшава"},
]
for _cc in EXTRA_CAPITAL_COUNTRIES:
    if _cc["country_qid"] not in {c["country_qid"] for c in CAPITAL_COUNTRIES}:
        CAPITAL_COUNTRIES.append(_cc)

EXTRA_STATIC_ANCHORS: List[Dict[str, Any]] = [
    {"qid": "Q13371", "en": "Harvard University", "ru": "Гарвардский университет", "country_qid": "Q30", "country_en": "United States", "country_ru": "США", "capital_qid": "Q61", "capital_en": "Washington, D.C.", "capital_ru": "Вашингтон", "year": 1636},
    {"qid": "Q49112", "en": "Yale University", "ru": "Йельский университет", "country_qid": "Q30", "country_en": "United States", "country_ru": "США", "capital_qid": "Q61", "capital_en": "Washington, D.C.", "capital_ru": "Вашингтон", "year": 1701},
    {"qid": "Q131252", "en": "University of Chicago", "ru": "Чикагский университет", "country_qid": "Q30", "country_en": "United States", "country_ru": "США", "capital_qid": "Q61", "capital_en": "Washington, D.C.", "capital_ru": "Вашингтон", "year": 1890},
    {"qid": "Q151510", "en": "Heidelberg University", "ru": "Гейдельбергский университет", "country_qid": "Q183", "country_en": "Germany", "country_ru": "Германия", "capital_qid": "Q64", "capital_en": "Berlin", "capital_ru": "Берлин", "year": 1386},
    {"qid": "Q152087", "en": "Humboldt University of Berlin", "ru": "Берлинский университет имени Гумбольдта", "country_qid": "Q183", "country_en": "Germany", "country_ru": "Германия", "capital_qid": "Q64", "capital_en": "Berlin", "capital_ru": "Берлин", "year": 1810},
    {"qid": "Q209842", "en": "University of Paris", "ru": "Парижский университет", "country_qid": "Q142", "country_en": "France", "country_ru": "Франция", "capital_qid": "Q90", "capital_en": "Paris", "capital_ru": "Париж", "year": 1150},
    {"qid": "Q7842", "en": "University of Tokyo", "ru": "Токийский университет", "country_qid": "Q17", "country_en": "Japan", "country_ru": "Япония", "capital_qid": "Q1490", "capital_en": "Tokyo", "capital_ru": "Токио", "year": 1877},
]
STATIC_ANCHORS.extend(EXTRA_STATIC_ANCHORS)
_anchor_by_qid = {a["qid"]: a for a in STATIC_ANCHORS}
for _a, _b in [("Q13371", "Q49112"), ("Q49112", "Q131252"), ("Q151510", "Q152087"), ("Q209842", "Q209842")]:
    if _a in _anchor_by_qid and _b in _anchor_by_qid and _a != _b:
        STATIC_ANCHOR_PAIRS.append((_anchor_by_qid[_a], _anchor_by_qid[_b]))

# Prefer countries where direct P31=university gold is not huge; this keeps
# complete gold lists instead of forcing broad/truncated records.
COUNTRIES = [c for c in COUNTRIES if c["qid"] in {cc["country_qid"] for cc in CAPITAL_COUNTRIES} | {"Q408"}]


def pick_static_anchor(rng: random.Random) -> Dict[str, Any]:
    return rng.choice(STATIC_ANCHORS)


def pick_static_anchor_pair(rng: random.Random) -> Tuple[Dict[str, Any], Dict[str, Any]]:
    return rng.choice(STATIC_ANCHOR_PAIRS)


def pick_capital_country(rng: random.Random) -> Dict[str, Any]:
    return rng.choice(CAPITAL_COUNTRIES)


def capital_country_lines(country: Dict[str, Any]) -> List[str]:
    return [
        f"?item wdt:P17 ?country .",
        f"?country wdt:P36 wd:{country['capital_qid']} .",
    ]


def static_same_country_lines(anchor: Dict[str, Any], *, exclude_anchor: bool = True) -> List[str]:
    lines = [
        f"wd:{anchor['qid']} wdt:P17 ?country .",
        "?item wdt:P17 ?country .",
    ]
    if exclude_anchor:
        lines.append(exclude_qid_line(anchor["qid"]))
    return lines

print(f"✅ curated L3-L5 pools ready: anchors={len(STATIC_ANCHORS)}, capital countries={len(CAPITAL_COUNTRIES)}")

YEAR_WINDOWS_L2: List[Tuple[int, int]] = [
    (1200, 1799), (1800, 1849), (1850, 1899), (1900, 1924),
    (1925, 1949), (1950, 1979), (1980, 1999), (2000, 2026),
]
YEAR_WINDOWS_L3: List[Tuple[int, int]] = [
    (1200, 1899), (1800, 1949), (1850, 1949), (1900, 1979), (1950, 2026),
]
YEAR_WINDOWS_L4: List[Tuple[int, int]] = [
    (1200, 1799), (1800, 1899), (1850, 1949), (1900, 1949), (1950, 1979), (1980, 2026),
]
YEAR_WINDOWS_L5: List[Tuple[int, int]] = [
    (1800, 1949), (1850, 1949), (1900, 1979), (1950, 2026),
]
REFERENCE_YEARS: List[int] = [1600, 1700, 1800, 1850, 1900, 1950, 1980, 2000]

print(f"✅ static pools ready: countries={len(COUNTRIES)}")


✅ curated L3-L5 pools ready: anchors=27, capital countries=22
✅ static pools ready: countries=23


In [15]:
# ============================================================
# 6. Lazy anchor, membership and admin pools


In [16]:
# ============================================================
_ANCHOR_POOL_DF: Optional[pd.DataFrame] = None
_MEMBERSHIP_POOL_DF: Optional[pd.DataFrame] = None
_ADMIN_POOL_DF: Optional[pd.DataFrame] = None


def normalize_pool_df(df: pd.DataFrame) -> pd.DataFrame:
    if df is None or len(df) == 0:
        return pd.DataFrame()
    out = df.copy()
    for col in out.columns:
        out[col] = out[col].fillna("").astype(str).map(clean_string)
    return out.drop_duplicates().reset_index(drop=True)


def build_anchor_pool() -> pd.DataFrame:
    # Direct P31=university keeps the anchor pool much cleaner than subclass closure.
    # Membership and admin fields are optional; templates requiring them filter later.
    sparql = f'''
    SELECT DISTINCT
      ?item ?itemLabelRu ?itemLabelEn
      ?country ?countryLabelRu ?countryLabelEn
      ?inception
      ?member ?memberLabelRu ?memberLabelEn
      ?admin ?adminLabelRu ?adminLabelEn
    WHERE {{
      ?item wdt:P31 wd:{Q_UNIVERSITY} .
      ?item wdt:P17 ?country .
      OPTIONAL {{ ?item wdt:P571 ?inception . }}
      OPTIONAL {{ ?item wdt:P463 ?member . }}
      OPTIONAL {{ ?item wdt:P131 ?admin . }}
      OPTIONAL {{ ?item rdfs:label ?itemLabelRu FILTER(LANG(?itemLabelRu) = "ru") . }}
      OPTIONAL {{ ?item rdfs:label ?itemLabelEn FILTER(LANG(?itemLabelEn) = "en") . }}
      OPTIONAL {{ ?country rdfs:label ?countryLabelRu FILTER(LANG(?countryLabelRu) = "ru") . }}
      OPTIONAL {{ ?country rdfs:label ?countryLabelEn FILTER(LANG(?countryLabelEn) = "en") . }}
      OPTIONAL {{ ?member rdfs:label ?memberLabelRu FILTER(LANG(?memberLabelRu) = "ru") . }}
      OPTIONAL {{ ?member rdfs:label ?memberLabelEn FILTER(LANG(?memberLabelEn) = "en") . }}
      OPTIONAL {{ ?admin rdfs:label ?adminLabelRu FILTER(LANG(?adminLabelRu) = "ru") . }}
      OPTIONAL {{ ?admin rdfs:label ?adminLabelEn FILTER(LANG(?adminLabelEn) = "en") . }}
      FILTER(BOUND(?itemLabelRu) || BOUND(?itemLabelEn)) .
    }}
    LIMIT 3500
    '''.strip()
    rows = rows_from_select(WD_CLIENT.sparql_select(sparql))
    records: List[Dict[str, Any]] = []
    for row in rows:
        item_qid = qid_from_any(row.get("item"))
        country_qid = qid_from_any(row.get("country"))
        member_qid = qid_from_any(row.get("member"))
        admin_qid = qid_from_any(row.get("admin"))
        item_en = en_name(row.get("itemLabelEn"), row.get("itemLabelRu"))
        item_ru = ru_name(row.get("itemLabelRu"), row.get("itemLabelEn"))
        country_en = en_name(row.get("countryLabelEn"), row.get("countryLabelRu"))
        country_ru = ru_name(row.get("countryLabelRu"), row.get("countryLabelEn"))
        member_en = en_name(row.get("memberLabelEn"), row.get("memberLabelRu"))
        member_ru = ru_name(row.get("memberLabelRu"), row.get("memberLabelEn"))
        admin_en = en_name(row.get("adminLabelEn"), row.get("adminLabelRu"))
        admin_ru = ru_name(row.get("adminLabelRu"), row.get("adminLabelEn"))
        inception_year = ""
        m = re.match(r"^(\d{3,4})", clean_string(row.get("inception", "")))
        if m:
            inception_year = m.group(1)
        if not item_qid or not country_qid or not item_en or not item_ru:
            continue
        records.append({
            "item_qid": item_qid,
            "item_label_ru": item_ru,
            "item_label_en": item_en,
            "country_qid": country_qid,
            "country_label_ru": country_ru,
            "country_label_en": country_en,
            "inception_year": inception_year,
            "member_qid": member_qid or "",
            "member_label_ru": member_ru,
            "member_label_en": member_en,
            "admin_qid": admin_qid or "",
            "admin_label_ru": admin_ru,
            "admin_label_en": admin_en,
        })
    return normalize_pool_df(pd.DataFrame(records))


def get_anchor_pool() -> pd.DataFrame:
    global _ANCHOR_POOL_DF
    if _ANCHOR_POOL_DF is None:
        _ANCHOR_POOL_DF = load_or_build_pool_safe("universities_anchor_pool_v8_direct", build_anchor_pool)
        _ANCHOR_POOL_DF = normalize_pool_df(_ANCHOR_POOL_DF)
        print(f"[pool] anchors: {len(_ANCHOR_POOL_DF)} rows")
    return _ANCHOR_POOL_DF


def build_membership_pool() -> pd.DataFrame:
    df = get_anchor_pool()
    if df is None or len(df) == 0 or "member_qid" not in df.columns:
        return pd.DataFrame()
    sub = df[df["member_qid"].str.fullmatch(r"Q\d+").fillna(False) & df["member_label_en"].ne("")].copy()
    if len(sub) == 0:
        return pd.DataFrame()
    counts = sub.groupby(["member_qid", "member_label_en", "member_label_ru"], dropna=False).size().reset_index(name="n")
    counts = counts[counts["n"] >= 3].copy()
    return normalize_pool_df(counts.sort_values(["n", "member_label_en"], ascending=[False, True]))


def get_membership_pool() -> pd.DataFrame:
    global _MEMBERSHIP_POOL_DF
    if _MEMBERSHIP_POOL_DF is None:
        _MEMBERSHIP_POOL_DF = load_or_build_pool_safe("universities_membership_pool_v8_direct", build_membership_pool)
        _MEMBERSHIP_POOL_DF = normalize_pool_df(_MEMBERSHIP_POOL_DF)
        print(f"[pool] memberships: {len(_MEMBERSHIP_POOL_DF)} rows")
    return _MEMBERSHIP_POOL_DF


def build_admin_pool() -> pd.DataFrame:
    df = get_anchor_pool()
    if df is None or len(df) == 0 or "admin_qid" not in df.columns:
        return pd.DataFrame()
    sub = df[df["admin_qid"].str.fullmatch(r"Q\d+").fillna(False) & df["admin_label_en"].ne("")].copy()
    if len(sub) == 0:
        return pd.DataFrame()
    counts = sub.groupby(["admin_qid", "admin_label_en", "admin_label_ru", "country_qid", "country_label_en", "country_label_ru"], dropna=False).size().reset_index(name="n")
    counts = counts[counts["n"] >= 3].copy()
    return normalize_pool_df(counts.sort_values(["n", "admin_label_en"], ascending=[False, True]))


def get_admin_pool() -> pd.DataFrame:
    global _ADMIN_POOL_DF
    if _ADMIN_POOL_DF is None:
        _ADMIN_POOL_DF = load_or_build_pool_safe("universities_admin_pool_v8_direct", build_admin_pool)
        _ADMIN_POOL_DF = normalize_pool_df(_ADMIN_POOL_DF)
        print(f"[pool] admin areas: {len(_ADMIN_POOL_DF)} rows")
    return _ADMIN_POOL_DF


def sample_df_row(df: pd.DataFrame, rng: random.Random) -> Optional[Dict[str, Any]]:
    if df is None or len(df) == 0:
        return None
    row = df.sample(1, random_state=rng.randint(0, 10**9)).iloc[0]
    return {k: row[k] for k in df.columns}


def pick_anchor(rng: random.Random, *, require_year: bool = False, require_member: bool = False, require_admin: bool = False) -> Optional[Dict[str, Any]]:
    df = get_anchor_pool()
    if df is None or len(df) == 0:
        return None
    sub = df.copy()
    if require_year:
        sub = sub[sub["inception_year"].str.fullmatch(r"\d{3,4}").fillna(False)]
    if require_member:
        sub = sub[sub["member_qid"].str.fullmatch(r"Q\d+").fillna(False)]
    if require_admin:
        sub = sub[sub["admin_qid"].str.fullmatch(r"Q\d+").fillna(False)]
    if len(sub) == 0:
        return None
    return sample_df_row(sub, rng)


def pick_membership(rng: random.Random, *, min_n: int = 3) -> Optional[Dict[str, Any]]:
    df = get_membership_pool()
    if df is None or len(df) == 0:
        return None
    sub = df.copy()
    if "n" in sub.columns:
        sub = sub[sub["n"].astype(str).astype(int) >= int(min_n)]
    return sample_df_row(sub, rng)


def pick_admin_area(rng: random.Random, *, min_n: int = 3) -> Optional[Dict[str, Any]]:
    df = get_admin_pool()
    if df is None or len(df) == 0:
        return None
    sub = df.copy()
    if "n" in sub.columns:
        sub = sub[sub["n"].astype(str).astype(int) >= int(min_n)]
    return sample_df_row(sub, rng)


def pick_country(rng: random.Random) -> Dict[str, str]:
    return rng.choice(COUNTRIES)

print("✅ lazy Wikidata pools ready")


✅ lazy Wikidata pools ready


In [17]:
# ============================================================


In [18]:
# ============================================================
# 7. Higher-quality relation helpers for L3-L5


In [19]:
# ============================================================

def static_same_country_lines_no_exclude(anchor: Dict[str, Any]) -> List[str]:
    return [
        f"wd:{anchor['qid']} wdt:P17 ?country .",
        "?item wdt:P17 ?country .",
    ]


def static_same_admin_lines_no_exclude(anchor: Dict[str, Any]) -> List[str]:
    return [
        f"wd:{anchor['qid']} wdt:P131 ?admin_area .",
        "?item wdt:P131 ?admin_area .",
    ]


def strict_between_anchor_years_lines(anchor_a: Dict[str, Any], anchor_b: Dict[str, Any]) -> Tuple[List[str], int, int]:
    low, high = sorted([int(anchor_a["year"]), int(anchor_b["year"])])
    # Strict interval avoids explicit "excluding" criteria while still preventing
    # the two anchors from becoming answers merely because their years define the range.
    return year_filter_lines(min_year=low + 1, max_year=high - 1), low + 1, high - 1


def nobel_alumni_lines(person_var: str = "nobel_person") -> List[str]:
    return [
        f"?{person_var} wdt:P69 ?item .",
        f"{{ ?{person_var} wdt:P166 wd:{Q_NOBEL_PRIZE} . }} UNION {{ ?{person_var} wdt:P166 ?nobel_award . ?nobel_award wdt:P279* wd:{Q_NOBEL_PRIZE} . }} UNION {{ ?{person_var} wdt:P166 ?nobel_award2 . ?nobel_award2 wdt:P31/wdt:P279* wd:{Q_NOBEL_PRIZE} . }}",
    ]


def nobel_alumni_born_in_country_lines(country_qid: str, person_var: str = "nobel_person") -> List[str]:
    return [
        *nobel_alumni_lines(person_var=person_var),
        f"?{person_var} wdt:P19 ?birth_place .",
        f"?birth_place wdt:P17 wd:{country_qid} .",
    ]


def membership_org_lines(org_qid: str) -> List[str]:
    return [f"?item wdt:P463 wd:{org_qid} ."]


def same_membership_as_anchor_lines_no_exclude(anchor: Dict[str, Any]) -> List[str]:
    return [
        f"wd:{anchor['qid']} wdt:P463 ?university_association .",
        "?item wdt:P463 ?university_association .",
    ]


def pick_membership_or_reject(rng: random.Random, *, min_n: int = 3) -> Optional[Dict[str, Any]]:
    org = pick_membership(rng, min_n=min_n)
    if not org:
        reject("membership_pool_empty")
        return None
    org_qid = qid_from_any(org.get("member_qid"))
    org_en = en_name(org.get("member_label_en"), org.get("member_label_ru"))
    org_ru = ru_name(org.get("member_label_ru"), org.get("member_label_en"))
    if not org_qid or not org_en or not org_ru:
        reject("bad_membership_pool_row")
        return None
    return {"qid": org_qid, "en": org_en, "ru": org_ru, "n": int(org.get("n") or 0)}


def pick_admin_anchor_or_reject(rng: random.Random) -> Optional[Dict[str, Any]]:
    anchor = pick_anchor(rng, require_year=True, require_admin=True)
    if not anchor:
        reject("admin_anchor_pool_empty")
        return None
    item_qid = qid_from_any(anchor.get("item_qid"))
    admin_qid = qid_from_any(anchor.get("admin_qid"))
    if not item_qid or not admin_qid:
        reject("bad_admin_anchor_pool_row")
        return None
    return {
        "qid": item_qid,
        "en": en_name(anchor.get("item_label_en"), anchor.get("item_label_ru")),
        "ru": ru_name(anchor.get("item_label_ru"), anchor.get("item_label_en")),
        "admin_qid": admin_qid,
        "admin_en": en_name(anchor.get("admin_label_en"), anchor.get("admin_label_ru")),
        "admin_ru": ru_name(anchor.get("admin_label_ru"), anchor.get("admin_label_en")),
        "year": int(anchor.get("inception_year") or 0),
    }

print("✅ v8 relation helpers ready: Nobel alumni, membership and admin-area patterns")


✅ v8 relation helpers ready: Nobel alumni, membership and admin-area patterns


In [20]:
# ============================================================
# 8. L1 templates: disabled by default


In [21]:
# ============================================================
# Country-only L1 records are intentionally not generated in v8. They were too
# easy and produced low-value benchmark examples. Keep an empty list so the
# schema and generation loop still support L1 if the target is manually changed.
L1_TEMPLATES: List[Callable[[int, random.Random], Optional[BenchmarkExample]]] = []
print(f"✅ L1 templates: {len(L1_TEMPLATES)} (disabled; target={TARGET_PER_LEVEL['L1']})")


✅ L1 templates: 0 (disabled; target=0)


In [22]:
# ============================================================
# 9. L2 templates: natural two-criterion filters


In [23]:
# ============================================================

def make_l2_country_inception_window(idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    country = pick_country(rng)
    y1, y2 = rng.choice(YEAR_WINDOWS_L2)
    k = REQUESTED_BY_LEVEL["L2"]
    where = [f"?item wdt:P17 wd:{country['qid']} .", *inception_window_lines(y1, y2)]
    return make_example(
        level="L2", idx=idx,
        query_text_ru=f"Назови {k} {ru_university_word(k)} страны «{country['ru']}», основанных в период {y1}–{y2}.",
        query_text_en=f"Name {k} universities in {country['en']} that were founded between {y1} and {y2}.",
        constraints={"kind": "university", "country": country["en"], "inception_year_from": y1, "inception_year_to": y2},
        where_lines=where,
        template_id="universities_l2_country_inception_window",
        template_family="country_year",
        is_advanced=False,
        constraint_entity_qids={"country": country["qid"]},
    )


def make_l2_country_before_year(idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    country = pick_country(rng)
    year = rng.choice(REFERENCE_YEARS)
    k = REQUESTED_BY_LEVEL["L2"]
    where = [f"?item wdt:P17 wd:{country['qid']} .", *before_year_lines(year)]
    return make_example(
        level="L2", idx=idx,
        query_text_ru=f"Назови {k} {ru_university_word(k)} страны «{country['ru']}», основанных раньше {year} года.",
        query_text_en=f"Name {k} universities in {country['en']} that were founded before {year}.",
        constraints={"kind": "university", "country": country["en"], "inception_year_before": year},
        where_lines=where,
        template_id="universities_l2_country_before_year",
        template_family="country_year",
        is_advanced=False,
        constraint_entity_qids={"country": country["qid"]},
    )


def make_l2_country_nobel_alumni(idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    country = pick_country(rng)
    k = REQUESTED_BY_LEVEL["L2"]
    where = [f"?item wdt:P17 wd:{country['qid']} .", *nobel_alumni_lines()]
    return make_example(
        level="L2", idx=idx,
        query_text_ru=f"Назови {k} {ru_university_word(k)} страны «{country['ru']}», где учился хотя бы один лауреат Нобелевской премии.",
        query_text_en=f"Name {k} universities in {country['en']} where at least one Nobel Prize laureate studied.",
        constraints={"kind": "university", "country": country["en"], "alumnus_award": "Nobel Prize"},
        where_lines=where,
        template_id="universities_l2_country_nobel_alumni",
        template_family="country_nobel_alumni",
        is_advanced=False,
        constraint_entity_qids={"country": country["qid"], "award": Q_NOBEL_PRIZE},
    )

L2_TEMPLATES = [make_l2_country_inception_window, make_l2_country_before_year, make_l2_country_nobel_alumni]
print(f"✅ L2 templates: {len(L2_TEMPLATES)}")


✅ L2 templates: 3


In [24]:
# ============================================================
# 10. L3 templates: real multihop, no country-only records


In [25]:
# ============================================================

def make_l3_same_country_after_anchor(idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    anchor = pick_static_anchor(rng)
    k = REQUESTED_BY_LEVEL["L3"]
    where = [*static_same_country_lines_no_exclude(anchor), *after_year_lines(anchor["year"])]
    return make_example(
        level="L3", idx=idx,
        query_text_ru=f"Назови {k} {ru_university_word(k)} из той же страны, что и «{anchor['ru']}», основанных позже этого университета.",
        query_text_en=f"Name {k} universities from the same country as {anchor['en']} that were founded later than that university.",
        constraints={"kind": "university", "country_from_university": anchor["en"], "founded_later_than_university": anchor["en"]},
        where_lines=where,
        template_id="universities_l3_same_country_after_anchor",
        template_family="anchor_country_year",
        is_advanced=True,
        constraint_entity_qids={"country_anchor_university": anchor["qid"], "year_anchor_university": anchor["qid"], "country": anchor["country_qid"]},
        derived_labels={"country": anchor["country_en"], "anchor_year": anchor["year"]},
    )


def make_l3_capital_country_nobel(idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    country = pick_capital_country(rng)
    k = REQUESTED_BY_LEVEL["L3"]
    where = [*capital_country_lines(country), *nobel_alumni_lines()]
    return make_example(
        level="L3", idx=idx,
        query_text_ru=f"Назови {k} {ru_university_word(k)} в стране, столицей которой является «{country['capital_ru']}», где учился хотя бы один лауреат Нобелевской премии.",
        query_text_en=f"Name {k} universities in the country whose capital is {country['capital_en']} where at least one Nobel Prize laureate studied.",
        constraints={"kind": "university", "country_capital": country["capital_en"], "alumnus_award": "Nobel Prize"},
        where_lines=where,
        template_id="universities_l3_capital_country_nobel_alumni",
        template_family="country_capital_nobel_alumni",
        is_advanced=True,
        constraint_entity_qids={"country": country["country_qid"], "capital": country["capital_qid"], "award": Q_NOBEL_PRIZE},
        derived_labels={"country": country["country_en"]},
    )


def make_l3_membership_before_year(idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    org = pick_membership_or_reject(rng, min_n=4)
    if not org:
        return None
    year = rng.choice([1850, 1900, 1950, 1980, 2000])
    k = REQUESTED_BY_LEVEL["L3"]
    where = [*membership_org_lines(org["qid"]), *before_year_lines(year)]
    return make_example(
        level="L3", idx=idx,
        query_text_ru=f"Назови {k} {ru_university_word(k)}, входящих в организацию «{org['ru']}» и основанных раньше {year} года.",
        query_text_en=f"Name {k} universities that are members of {org['en']} and were founded before {year}.",
        constraints={"kind": "university", "member_of": org["en"], "inception_year_before": year},
        where_lines=where,
        template_id="universities_l3_membership_before_year",
        template_family="membership_year",
        is_advanced=True,
        constraint_entity_qids={"membership": org["qid"]},
    )


def make_l3_nobel_alumni_after_year(idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    year = rng.choice([1600, 1700, 1800, 1850, 1900, 1950])
    k = REQUESTED_BY_LEVEL["L3"]
    where = [*nobel_alumni_lines(), *after_year_lines(year)]
    return make_example(
        level="L3", idx=idx,
        query_text_ru=f"Назови {k} {ru_university_word(k)}, основанных позже {year} года, где учился хотя бы один лауреат Нобелевской премии.",
        query_text_en=f"Name {k} universities founded after {year} where at least one Nobel Prize laureate studied.",
        constraints={"kind": "university", "inception_year_after": year, "alumnus_award": "Nobel Prize"},
        where_lines=where,
        template_id="universities_l3_nobel_alumni_after_year",
        template_family="nobel_alumni_year",
        is_advanced=True,
        constraint_entity_qids={"award": Q_NOBEL_PRIZE},
    )

L3_TEMPLATES = [
    make_l3_same_country_after_anchor,
    make_l3_capital_country_nobel,
    make_l3_membership_before_year,
    make_l3_nobel_alumni_after_year,
]
print(f"✅ L3 templates: {len(L3_TEMPLATES)}")


✅ L3 templates: 4


In [26]:
# ============================================================
# 11. L4 templates: richer multi-criteria multihops


In [27]:
# ============================================================

def make_l4_same_country_nobel_after_anchor(idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    anchor = pick_static_anchor(rng)
    k = REQUESTED_BY_LEVEL["L4"]
    where = [*static_same_country_lines_no_exclude(anchor), *nobel_alumni_lines(), *after_year_lines(anchor["year"])]
    return make_example(
        level="L4", idx=idx,
        query_text_ru=f"Назови {k} {ru_university_word(k)} из той же страны, что и «{anchor['ru']}», основанных позже него, где учился хотя бы один лауреат Нобелевской премии.",
        query_text_en=f"Name {k} universities from the same country as {anchor['en']} that were founded later than it and where at least one Nobel Prize laureate studied.",
        constraints={"kind": "university", "country_from_university": anchor["en"], "founded_later_than_university": anchor["en"], "alumnus_award": "Nobel Prize"},
        where_lines=where,
        template_id="universities_l4_anchor_country_nobel_after_anchor",
        template_family="anchor_country_year_nobel",
        is_advanced=True,
        constraint_entity_qids={"country_anchor_university": anchor["qid"], "year_anchor_university": anchor["qid"], "country": anchor["country_qid"], "award": Q_NOBEL_PRIZE},
        derived_labels={"country": anchor["country_en"], "anchor_year": anchor["year"]},
    )


def make_l4_capital_country_nobel_window(idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    country = pick_capital_country(rng)
    y1, y2 = rng.choice(YEAR_WINDOWS_L4)
    k = REQUESTED_BY_LEVEL["L4"]
    where = [*capital_country_lines(country), *nobel_alumni_lines(), *inception_window_lines(y1, y2)]
    return make_example(
        level="L4", idx=idx,
        query_text_ru=f"Назови {k} {ru_university_word(k)} в стране, столицей которой является «{country['capital_ru']}», основанных в период {y1}–{y2}, где учился хотя бы один лауреат Нобелевской премии.",
        query_text_en=f"Name {k} universities in the country whose capital is {country['capital_en']}, founded between {y1} and {y2}, where at least one Nobel Prize laureate studied.",
        constraints={"kind": "university", "country_capital": country["capital_en"], "inception_year_from": y1, "inception_year_to": y2, "alumnus_award": "Nobel Prize"},
        where_lines=where,
        template_id="universities_l4_capital_country_nobel_window",
        template_family="country_capital_year_nobel",
        is_advanced=True,
        constraint_entity_qids={"country": country["country_qid"], "capital": country["capital_qid"], "award": Q_NOBEL_PRIZE},
        derived_labels={"country": country["country_en"]},
    )


def make_l4_country_membership_year(idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    country = pick_country(rng)
    org = pick_membership_or_reject(rng, min_n=4)
    if not org:
        return None
    year = rng.choice([1850, 1900, 1950, 1980, 2000])
    k = REQUESTED_BY_LEVEL["L4"]
    where = [f"?item wdt:P17 wd:{country['qid']} .", *membership_org_lines(org["qid"]), *after_year_lines(year)]
    return make_example(
        level="L4", idx=idx,
        query_text_ru=f"Назови {k} {ru_university_word(k)} страны «{country['ru']}», входящих в организацию «{org['ru']}» и основанных позже {year} года.",
        query_text_en=f"Name {k} universities in {country['en']} that are members of {org['en']} and were founded after {year}.",
        constraints={"kind": "university", "country": country["en"], "member_of": org["en"], "inception_year_after": year},
        where_lines=where,
        template_id="universities_l4_country_membership_after_year",
        template_family="country_membership_year",
        is_advanced=True,
        constraint_entity_qids={"country": country["qid"], "membership": org["qid"]},
    )


def make_l4_same_admin_nobel_after_anchor(idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    anchor = pick_admin_anchor_or_reject(rng)
    if not anchor or not anchor.get("year"):
        return None
    k = REQUESTED_BY_LEVEL["L4"]
    where = [
        f"wd:{anchor['qid']} wdt:P131 ?admin_area .",
        "?item wdt:P131 ?admin_area .",
        *nobel_alumni_lines(),
        *after_year_lines(anchor["year"]),
    ]
    return make_example(
        level="L4", idx=idx,
        query_text_ru=f"Назови {k} {ru_university_word(k)} в той же административной единице, что и «{anchor['ru']}», основанных позже него, где учился хотя бы один лауреат Нобелевской премии.",
        query_text_en=f"Name {k} universities in the same administrative territorial entity as {anchor['en']} that were founded later than it and where at least one Nobel Prize laureate studied.",
        constraints={"kind": "university", "admin_area_from_university": anchor["en"], "founded_later_than_university": anchor["en"], "alumnus_award": "Nobel Prize"},
        where_lines=where,
        template_id="universities_l4_admin_area_nobel_after_anchor",
        template_family="admin_area_year_nobel",
        is_advanced=True,
        constraint_entity_qids={"admin_anchor_university": anchor["qid"], "admin_area": anchor["admin_qid"], "award": Q_NOBEL_PRIZE},
        derived_labels={"admin_area": anchor["admin_en"], "anchor_year": anchor["year"]},
    )


def make_l4_nobel_birth_country_study_country(idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    study_country = pick_country(rng)
    birth_country = pick_country(rng)
    if study_country["qid"] == birth_country["qid"]:
        birth_country = pick_country(rng)
    k = REQUESTED_BY_LEVEL["L4"]
    where = [f"?item wdt:P17 wd:{study_country['qid']} .", *nobel_alumni_born_in_country_lines(birth_country["qid"])]
    return make_example(
        level="L4", idx=idx,
        query_text_ru=f"Назови {k} {ru_university_word(k)} страны «{study_country['ru']}», где учился лауреат Нобелевской премии, родившийся в стране «{birth_country['ru']}».",
        query_text_en=f"Name {k} universities in {study_country['en']} where a Nobel Prize laureate born in {birth_country['en']} studied.",
        constraints={"kind": "university", "country": study_country["en"], "alumnus_award": "Nobel Prize", "nobel_laureate_birth_country": birth_country["en"]},
        where_lines=where,
        template_id="universities_l4_nobel_birth_country_study_country",
        template_family="country_nobel_birth_country",
        is_advanced=True,
        constraint_entity_qids={"study_country": study_country["qid"], "birth_country": birth_country["qid"], "award": Q_NOBEL_PRIZE},
    )

L4_TEMPLATES = [
    make_l4_same_country_nobel_after_anchor,
    make_l4_capital_country_nobel_window,
    make_l4_country_membership_year,
    make_l4_same_admin_nobel_after_anchor,
    make_l4_nobel_birth_country_study_country,
]
print(f"✅ L4 templates: {len(L4_TEMPLATES)}")


✅ L4 templates: 5


In [28]:
# ============================================================
# 12. L5 templates: genuinely hard multihop and multi-criterion


In [29]:
# ============================================================

def make_l5_same_country_between_anchor_years_nobel(idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    anchor_a, anchor_b = pick_static_anchor_pair(rng)
    lines, y1, y2 = strict_between_anchor_years_lines(anchor_a, anchor_b)
    if y1 > y2:
        return None
    k = REQUESTED_BY_LEVEL["L5"]
    where = [*static_same_country_lines_no_exclude(anchor_a), *lines, *nobel_alumni_lines()]
    return make_example(
        level="L5", idx=idx,
        query_text_ru=f"Назови {k} {ru_university_word(k)} из той же страны, что и «{anchor_a['ru']}», основанных после года основания «{anchor_a['ru']}» и до года основания «{anchor_b['ru']}», где учился хотя бы один лауреат Нобелевской премии.",
        query_text_en=f"Name {k} universities from the same country as {anchor_a['en']} that were founded after the founding year of {anchor_a['en']} and before the founding year of {anchor_b['en']}, and where at least one Nobel Prize laureate studied.",
        constraints={"kind": "university", "country_from_university": anchor_a["en"], "founded_after_university": anchor_a["en"], "founded_before_university": anchor_b["en"], "alumnus_award": "Nobel Prize"},
        where_lines=where,
        template_id="universities_l5_anchor_country_between_years_nobel",
        template_family="two_anchor_country_year_nobel",
        is_advanced=True,
        constraint_entity_qids={"country_anchor_university": anchor_a["qid"], "after_year_university": anchor_a["qid"], "before_year_university": anchor_b["qid"], "country": anchor_a["country_qid"], "award": Q_NOBEL_PRIZE},
        derived_labels={"country": anchor_a["country_en"], "year_from": y1, "year_to": y2},
    )


def make_l5_capital_country_membership_nobel_window(idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    country = pick_capital_country(rng)
    org = pick_membership_or_reject(rng, min_n=4)
    if not org:
        return None
    y1, y2 = rng.choice(YEAR_WINDOWS_L5)
    k = REQUESTED_BY_LEVEL["L5"]
    where = [*capital_country_lines(country), *membership_org_lines(org["qid"]), *nobel_alumni_lines(), *inception_window_lines(y1, y2)]
    return make_example(
        level="L5", idx=idx,
        query_text_ru=f"Назови {k} {ru_university_word(k)} в стране, столицей которой является «{country['capital_ru']}», входящих в организацию «{org['ru']}», основанных в период {y1}–{y2}, где учился хотя бы один лауреат Нобелевской премии.",
        query_text_en=f"Name {k} universities in the country whose capital is {country['capital_en']}, that are members of {org['en']}, were founded between {y1} and {y2}, and where at least one Nobel Prize laureate studied.",
        constraints={"kind": "university", "country_capital": country["capital_en"], "member_of": org["en"], "inception_year_from": y1, "inception_year_to": y2, "alumnus_award": "Nobel Prize"},
        where_lines=where,
        template_id="universities_l5_capital_country_membership_nobel_window",
        template_family="country_capital_membership_year_nobel",
        is_advanced=True,
        constraint_entity_qids={"country": country["country_qid"], "capital": country["capital_qid"], "membership": org["qid"], "award": Q_NOBEL_PRIZE},
        derived_labels={"country": country["country_en"]},
    )


def make_l5_country_membership_nobel_birth_country(idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    study_country = pick_country(rng)
    birth_country = pick_country(rng)
    if study_country["qid"] == birth_country["qid"]:
        birth_country = pick_country(rng)
    org = pick_membership_or_reject(rng, min_n=4)
    if not org:
        return None
    k = REQUESTED_BY_LEVEL["L5"]
    where = [f"?item wdt:P17 wd:{study_country['qid']} .", *membership_org_lines(org["qid"]), *nobel_alumni_born_in_country_lines(birth_country["qid"])]
    return make_example(
        level="L5", idx=idx,
        query_text_ru=f"Назови {k} {ru_university_word(k)} страны «{study_country['ru']}», входящих в организацию «{org['ru']}», где учился лауреат Нобелевской премии, родившийся в стране «{birth_country['ru']}».",
        query_text_en=f"Name {k} universities in {study_country['en']} that are members of {org['en']} and where a Nobel Prize laureate born in {birth_country['en']} studied.",
        constraints={"kind": "university", "country": study_country["en"], "member_of": org["en"], "alumnus_award": "Nobel Prize", "nobel_laureate_birth_country": birth_country["en"]},
        where_lines=where,
        template_id="universities_l5_country_membership_nobel_birth_country",
        template_family="country_membership_nobel_birth_country",
        is_advanced=True,
        constraint_entity_qids={"study_country": study_country["qid"], "birth_country": birth_country["qid"], "membership": org["qid"], "award": Q_NOBEL_PRIZE},
    )


def make_l5_same_membership_as_anchor_nobel_after_anchor(idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    anchor = pick_anchor(rng, require_year=True, require_member=True)
    if not anchor:
        reject("membership_anchor_pool_empty")
        return None
    anchor_qid = qid_from_any(anchor.get("item_qid"))
    anchor_en = en_name(anchor.get("item_label_en"), anchor.get("item_label_ru"))
    anchor_ru = ru_name(anchor.get("item_label_ru"), anchor.get("item_label_en"))
    anchor_year = int(anchor.get("inception_year") or 0)
    member_qid = qid_from_any(anchor.get("member_qid"))
    member_en = en_name(anchor.get("member_label_en"), anchor.get("member_label_ru"))
    member_ru = ru_name(anchor.get("member_label_ru"), anchor.get("member_label_en"))
    if not anchor_qid or not member_qid or not anchor_year or not member_en or not member_ru:
        reject("bad_membership_anchor_pool_row")
        return None
    k = REQUESTED_BY_LEVEL["L5"]
    where = [*same_membership_as_anchor_lines_no_exclude({"qid": anchor_qid}), *nobel_alumni_lines(), *after_year_lines(anchor_year)]
    return make_example(
        level="L5", idx=idx,
        query_text_ru=f"Назови {k} {ru_university_word(k)}, входящих в ту же организацию, что и «{anchor_ru}», основанных позже него, где учился хотя бы один лауреат Нобелевской премии.",
        query_text_en=f"Name {k} universities that are members of the same organization as {anchor_en}, were founded later than it, and where at least one Nobel Prize laureate studied.",
        constraints={"kind": "university", "membership_from_university": anchor_en, "founded_later_than_university": anchor_en, "alumnus_award": "Nobel Prize"},
        where_lines=where,
        template_id="universities_l5_same_membership_nobel_after_anchor",
        template_family="anchor_membership_year_nobel",
        is_advanced=True,
        constraint_entity_qids={"membership_anchor_university": anchor_qid, "membership": member_qid, "award": Q_NOBEL_PRIZE},
        derived_labels={"membership": member_en, "anchor_year": anchor_year},
    )


def make_l5_admin_area_membership_nobel_after_anchor(idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    anchor = pick_admin_anchor_or_reject(rng)
    if not anchor or not anchor.get("year"):
        return None
    org = pick_membership_or_reject(rng, min_n=4)
    if not org:
        return None
    k = REQUESTED_BY_LEVEL["L5"]
    where = [
        f"wd:{anchor['qid']} wdt:P131 ?admin_area .",
        "?item wdt:P131 ?admin_area .",
        *membership_org_lines(org["qid"]),
        *nobel_alumni_lines(),
        *after_year_lines(anchor["year"]),
    ]
    return make_example(
        level="L5", idx=idx,
        query_text_ru=f"Назови {k} {ru_university_word(k)} в той же административной единице, что и «{anchor['ru']}», входящих в организацию «{org['ru']}», основанных позже этого университета, где учился хотя бы один лауреат Нобелевской премии.",
        query_text_en=f"Name {k} universities in the same administrative territorial entity as {anchor['en']}, that are members of {org['en']}, were founded later than that university, and where at least one Nobel Prize laureate studied.",
        constraints={"kind": "university", "admin_area_from_university": anchor["en"], "member_of": org["en"], "founded_later_than_university": anchor["en"], "alumnus_award": "Nobel Prize"},
        where_lines=where,
        template_id="universities_l5_admin_membership_nobel_after_anchor",
        template_family="admin_membership_year_nobel",
        is_advanced=True,
        constraint_entity_qids={"admin_anchor_university": anchor["qid"], "admin_area": anchor["admin_qid"], "membership": org["qid"], "award": Q_NOBEL_PRIZE},
        derived_labels={"admin_area": anchor["admin_en"], "anchor_year": anchor["year"]},
    )



def make_l5_country_nobel_birth_country_after_year(idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    # High-yield hard fallback: university country + Nobel alumnus + laureate birth country + founding year.
    study_country = rng.choice([c for c in COUNTRIES if c["qid"] in {"Q30", "Q145", "Q183", "Q142", "Q17", "Q39", "Q40", "Q55", "Q38", "Q16", "Q408"}])
    birth_country = rng.choice([c for c in COUNTRIES if c["qid"] in {"Q30", "Q145", "Q183", "Q142", "Q17", "Q39", "Q40", "Q55", "Q38", "Q34", "Q36"}])
    if study_country["qid"] == birth_country["qid"]:
        birth_country = rng.choice([c for c in COUNTRIES if c["qid"] != study_country["qid"]])
    year = rng.choice([1600, 1700, 1800, 1850, 1900, 1950])
    k = REQUESTED_BY_LEVEL["L5"]
    where = [f"?item wdt:P17 wd:{study_country['qid']} .", *nobel_alumni_born_in_country_lines(birth_country["qid"]), *after_year_lines(year)]
    return make_example(
        level="L5", idx=idx,
        query_text_ru=f"Назови {k} {ru_university_word(k)} страны «{study_country['ru']}», основанных позже {year} года, где учился лауреат Нобелевской премии, родившийся в стране «{birth_country['ru']}».",
        query_text_en=f"Name {k} universities in {study_country['en']} that were founded after {year} and where a Nobel Prize laureate born in {birth_country['en']} studied.",
        constraints={"kind": "university", "country": study_country["en"], "inception_year_after": year, "alumnus_award": "Nobel Prize", "nobel_laureate_birth_country": birth_country["en"]},
        where_lines=where,
        template_id="universities_l5_country_nobel_birth_country_after_year",
        template_family="country_year_nobel_birth_country",
        is_advanced=True,
        constraint_entity_qids={"study_country": study_country["qid"], "birth_country": birth_country["qid"], "award": Q_NOBEL_PRIZE},
    )

L5_TEMPLATES = [
    make_l5_country_nobel_birth_country_after_year,
    make_l5_same_country_between_anchor_years_nobel,
    make_l5_capital_country_membership_nobel_window,
    make_l5_country_membership_nobel_birth_country,
    make_l5_same_membership_as_anchor_nobel_after_anchor,
    make_l5_admin_area_membership_nobel_after_anchor,
]
print(f"✅ L5 templates: {len(L5_TEMPLATES)}")

# 12. Generation loop


✅ L5 templates: 6


In [30]:
# ============================================================
TEMPLATES_BY_LEVEL: Dict[str, List[Callable[[int, random.Random], Optional[BenchmarkExample]]]] = {
    "L1": L1_TEMPLATES,
    "L2": L2_TEMPLATES,
    "L3": L3_TEMPLATES,
    "L4": L4_TEMPLATES,
    "L5": L5_TEMPLATES,
}


def semantic_key(ex: BenchmarkExample) -> Tuple[Any, ...]:
    return (ex.domain, ex.complexity, ex.template_id, json.dumps(ex.constraints, ensure_ascii=False, sort_keys=True))


def gold_key(ex: BenchmarkExample) -> Tuple[str, ...]:
    return tuple(sorted(ex.gold_answer_qids))


def validate_example_basic(ex: BenchmarkExample) -> List[str]:
    errors: List[str] = []
    obj = ordered_as_benchmark_example(ex)
    if list(obj.keys()) != EXPECTED_KEYS:
        errors.append("schema_key_order_mismatch")
    if ex.domain != DOMAIN:
        errors.append("wrong_domain")
    if ex.complexity not in LEVELS:
        errors.append("wrong_complexity")
    if not question_text_ok(ex.query_text_ru) or not question_text_ok(ex.query_text_en):
        errors.append("bad_query_text")
    if len(ex.gold_answer_qids) == 0:
        errors.append("zero_gold")
    if len(ex.gold_answer_qids) < ex.requested_count:
        errors.append("gold_less_than_requested")
    if len(ex.gold_answer_qids) != len(ex.gold_answer_labels_ru):
        errors.append("ru_gold_label_length_mismatch")
    if len(ex.gold_answer_qids) != len(ex.gold_answer_labels_en):
        errors.append("en_gold_label_length_mismatch")
    if ex.gold_truncated:
        errors.append("gold_truncated_true")
    if "wd:{ITEM}" not in (ex.ask_validator_sparql or ""):
        errors.append("ask_validator_missing_placeholder")
    if not constraints_are_clean(ex.constraints):
        errors.append("bad_constraints")

    semantic_criteria = [k for k in (ex.constraints or {}) if k != "kind"]
    if ex.complexity in {"L4", "L5"} and len(semantic_criteria) < 3:
        errors.append("too_few_semantic_constraints_for_level")
    if any(QID_ONLY_RE.fullmatch(str(x or "")) for x in ex.gold_answer_labels_ru):
        errors.append("qid_as_ru_label")
    if any(QID_ONLY_RE.fullmatch(str(x or "")) for x in ex.gold_answer_labels_en):
        errors.append("qid_as_en_label")
    return errors


def generate_one(level: str, idx: int, rng: random.Random) -> Optional[BenchmarkExample]:
    templates = TEMPLATES_BY_LEVEL[level]
    template = rng.choice(templates)
    return template(idx, rng)


def generate_universities_dataset(
    *,
    out_path: Path = OUTPUT_PATH,
    target_per_level: Dict[str, int] = TARGET_PER_LEVEL,
    seed: int = SEED,
    max_attempts_per_level: int = MAX_ATTEMPTS_PER_LEVEL,
    overwrite: bool = OVERWRITE_OUTPUT,
) -> List[BenchmarkExample]:
    rng = random.Random(seed)
    if overwrite and out_path.exists():
        out_path.unlink()

    examples: List[BenchmarkExample] = []
    seen_semantic = set()
    seen_gold = set()
    reject_counts: Counter = Counter()

    for level in LEVELS:
        target = int(target_per_level.get(level, 0))
        accepted = 0
        attempts = 0
        while accepted < target and attempts < max_attempts_per_level:
            attempts += 1
            idx = accepted + 1
            global LAST_REJECT_REASON
            LAST_REJECT_REASON = "none"
            try:
                ex = generate_one(level, idx, rng)
            except Exception as e:
                reject_counts[(level, "exception")] += 1
                if DEBUG_GENERATOR_ERRORS:
                    print(f"[WARN] {level} attempt {attempts}: {e}")
                continue
            if ex is None:
                reject_counts[(level, LAST_REJECT_REASON or "none")] += 1
                continue
            errors = validate_example_basic(ex)
            if errors:
                reject_counts[(level, ",".join(errors))] += 1
                continue
            sk = semantic_key(ex)
            gk = gold_key(ex)
            if sk in seen_semantic:
                reject_counts[(level, "duplicate_semantic")] += 1
                continue
            if gk in seen_gold:
                reject_counts[(level, "duplicate_gold_set")] += 1
                continue
            seen_semantic.add(sk)
            seen_gold.add(gk)
            examples.append(ex)
            accepted += 1
            append_example_jsonl(out_path, ex)
            print(f"[{DOMAIN}] {level}: {accepted}/{target} saved — {ex.template_id} — gold={len(ex.gold_answer_qids)}")
        if accepted < target:
            print(f"[WARN] {DOMAIN}:{level} generated {accepted}/{target} after {attempts} attempts")
            print("[WARN] common reject reasons:", reject_counts.most_common(12))
    print(f"✅ generation finished: {len(examples)} examples saved to {out_path}")
    return examples

print("✅ generation loop ready")


✅ generation loop ready


In [31]:
# ============================================================
# 13. Final JSONL validation


In [32]:
# ============================================================

def validate_output_file(path: Path = OUTPUT_PATH) -> Dict[str, Any]:
    rows = read_jsonl(path)
    report: Dict[str, Any] = {
        "path": str(path),
        "total": len(rows),
        "by_level": dict(Counter(row.get("complexity") for row in rows)),
        "errors": [],
        "warnings": [],
        "template_counts": dict(Counter(row.get("template_id") for row in rows)),
    }
    ids = set()
    semantic_seen = set()
    gold_seen = set()

    for i, row in enumerate(rows, 1):
        prefix = f"line_{i}:{row.get('id', '<no id>')}"
        if list(row.keys()) != EXPECTED_KEYS:
            report["errors"].append(f"{prefix}:schema_key_order_mismatch")
        rid = row.get("id")
        if rid in ids:
            report["errors"].append(f"{prefix}:duplicate_id")
        ids.add(rid)
        if row.get("domain") != DOMAIN:
            report["errors"].append(f"{prefix}:wrong_domain")
        if not question_text_ok(row.get("query_text_ru", "")) or not question_text_ok(row.get("query_text_en", "")):
            report["errors"].append(f"{prefix}:bad_query_text")
        constraints = row.get("constraints") or {}
        if not constraints_are_clean(constraints):
            report["errors"].append(f"{prefix}:bad_constraints")
        semantic_criteria = [k for k in constraints if k != "kind"]
        if row.get("complexity") in {"L4", "L5"} and len(semantic_criteria) < 3:
            report["errors"].append(f"{prefix}:too_few_semantic_constraints_for_level")
        requested = int(row.get("requested_count") or 0)
        qids = row.get("gold_answer_qids") or []
        ru_labels = row.get("gold_answer_labels_ru") or []
        en_labels = row.get("gold_answer_labels_en") or []
        if not qids:
            report["errors"].append(f"{prefix}:zero_gold")
        if len(qids) < requested:
            report["errors"].append(f"{prefix}:gold_less_than_requested")
        if len(qids) != len(ru_labels):
            report["errors"].append(f"{prefix}:ru_label_length_mismatch")
        if len(qids) != len(en_labels):
            report["errors"].append(f"{prefix}:en_label_length_mismatch")
        if row.get("gold_truncated"):
            report["errors"].append(f"{prefix}:gold_truncated_true")
        if "wd:{ITEM}" not in str(row.get("ask_validator_sparql") or ""):
            report["errors"].append(f"{prefix}:ask_validator_missing_placeholder")
        if any(QID_ONLY_RE.fullmatch(str(x or "")) for x in ru_labels):
            report["errors"].append(f"{prefix}:qid_as_ru_label")
        if any(QID_ONLY_RE.fullmatch(str(x or "")) for x in en_labels):
            report["errors"].append(f"{prefix}:qid_as_en_label")
        meta = row.get("gold_collection_meta") or {}
        if not isinstance(meta, dict) or not meta.get("constraint_entity_qids"):
            report["warnings"].append(f"{prefix}:missing_constraint_entity_qids_meta")
        sk = (row.get("domain"), row.get("complexity"), row.get("template_id"), json.dumps(constraints, ensure_ascii=False, sort_keys=True))
        if sk in semantic_seen:
            report["warnings"].append(f"{prefix}:duplicate_semantic_key")
        semantic_seen.add(sk)
        gk = tuple(sorted(qids))
        if gk in gold_seen:
            report["warnings"].append(f"{prefix}:duplicate_gold_set")
        gold_seen.add(gk)

    for level, target in TARGET_PER_LEVEL.items():
        actual = int(report["by_level"].get(level, 0) or 0)
        if actual != int(target):
            report["errors"].append(f"level_count_mismatch:{level}:{actual}!={target}")

    report_path = path.with_suffix(".validation_report.json")
    with report_path.open("w", encoding="utf-8") as f:
        json.dump(report, f, ensure_ascii=False, indent=2)
    print(json.dumps(report, ensure_ascii=False, indent=2)[:5000])
    print(f"✅ validation report saved to {report_path}")
    return report


def show_sample(path: Path = OUTPUT_PATH, n: int = 5) -> None:
    rows = read_jsonl(path)
    for row in rows[:n]:
        print("=" * 80)
        print(row.get("id"), row.get("complexity"), row.get("template_id"))
        print(row.get("query_text_ru"))
        print(row.get("query_text_en"))
        print("constraints:", json.dumps(row.get("constraints"), ensure_ascii=False))
        print("meta.constraint_entity_qids:", (row.get("gold_collection_meta") or {}).get("constraint_entity_qids"))
        print("gold:", len(row.get("gold_answer_qids") or []), row.get("gold_answer_labels_en", [])[:5])

print("✅ validation helpers ready")


✅ validation helpers ready


In [33]:
# ============================================================
# 14. Run generation


In [34]:
# ============================================================
if RUN_GENERATION:
    examples = generate_universities_dataset(
        out_path=OUTPUT_PATH,
        target_per_level=TARGET_PER_LEVEL,
        seed=SEED,
        max_attempts_per_level=MAX_ATTEMPTS_PER_LEVEL,
        overwrite=OVERWRITE_OUTPUT,
    )
    if RUN_FINAL_VALIDATION:
        validation_report = validate_output_file(OUTPUT_PATH)
        if validation_report.get("errors"):
            raise RuntimeError(f"Validation failed with {len(validation_report['errors'])} errors. See {OUTPUT_PATH.with_suffix('.validation_report.json')}")
        show_sample(OUTPUT_PATH, n=5)
else:
    print("RUN_GENERATION=False; set it to True to build universities.jsonl")


[universities] L2: 1/15 saved — universities_l2_country_before_year — gold=30
[universities] L2: 2/15 saved — universities_l2_country_before_year — gold=5
[universities] L2: 3/15 saved — universities_l2_country_inception_window — gold=11
[universities] L2: 4/15 saved — universities_l2_country_before_year — gold=9
[universities] L2: 5/15 saved — universities_l2_country_inception_window — gold=5
[universities] L2: 6/15 saved — universities_l2_country_before_year — gold=5
[universities] L2: 7/15 saved — universities_l2_country_inception_window — gold=21
[universities] L2: 8/15 saved — universities_l2_country_nobel_alumni — gold=38
[universities] L2: 9/15 saved — universities_l2_country_before_year — gold=7
[universities] L2: 10/15 saved — universities_l2_country_nobel_alumni — gold=7
[universities] L2: 11/15 saved — universities_l2_country_before_year — gold=7
[universities] L2: 12/15 saved — universities_l2_country_inception_window — gold=5
[universities] L2: 13/15 saved — universities_l

RuntimeError: Validation failed with 2 errors. See out_wikidata_benchmark/domain_outputs/universities.validation_report.json